In [ ]:
## Imports and code: 

%load_ext autoreload
%autoreload 2


import torch
import mlflow
import torch
import torch.nn as nn

from ra_utils.data.dataloader_CR_patches import (
    process_several_score_groups,
    dataset_and_loader_several,
    check_duplicates_in_dataloader
)

from ra_utils.training.scores_SHS.scores_SHS_training_lib_AE_v1 import (
    evaluate_and_log_testset_results_AE_v3,
    train_loop_AE_v3
)
from ra_utils.training.scores_SHS.scores_SHS_training_lib_AE_v4 import (
    evaluate_and_log_testset_results_AE_v4,
    train_loop_AE_v4
)

from ra_utils.networks.loss_function import get_score_loss_function, get_triplet_loss_fn
import torchvision.transforms.v2 as v2
from ra_utils.training.scores_SHS.model_builders import build_models_AE_v1_and2
import ra_utils.utils.utils_torch
from ra_utils.utils.verbosity_enums import *
import ra_utils.utils.utils

import ra_utils.utils.utils_torch
from pprint import pprint
import ra_utils.utils.config_parser

from ra_utils.utils.utils import datestr_to_years_since_2000


import numpy as np


from ra_utils.training.scores_SHS.run_training_main_lib import (
    check_config_consistency_and_partially_make_consistent,
    maybe_partially_init_model_from_state_dict,
)

import ra_utils.training.scores_SHS.run_training_main_lib

import matplotlib.pyplot as plt

In [ ]:
config, config_name, config_originals = ra_utils.utils.config_parser.load_config(
    default_config="/home/cwatzenboeck/code/RA/ra_utils/runs/config_scoring/development_inputs/training_confing.yml",
    debugging_in_jupyter_nb=True, 
    silencium=False, 
    return_config_name=True, 
    return_originals=True
    )

# print("PIPII: ", config["data"]["classifier_head_infos"]["PIPII"])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
#-----------------------------------------------------------------------
# Load tables with paths and scores (+ split)
# data_tables = load_img_SHS_patch_data(config["data"])
data_tables = process_several_score_groups(config["data"])

# Check that elements exist
for k,v  in data_tables.items():
    assert len(v["df_include"]) >0, f"score group {k} has no items. Check the image paths or score groups!"


# Make dataset and dataloaders
# data = dataset_and_loader(data_tables, config)
data = dataset_and_loader_several(data_tables, config)

if config.get("CheckDL4Duplicates", False): 
    # check_duplicates_in_dataloader(data, ds_key="train_loader")
    print("Checking for duplicates")
    check_duplicates_in_dataloader(data, ds_key="val_loader")
    check_duplicates_in_dataloader(data, ds_key="test_loader")
    print("Done - Checking for duplicates\n ")


In [ ]:


# Load some sample data: 
dl = data["ALL"]["train_loader"]
batch = next(iter(dl))
print(batch.keys())



X      = batch["img"].to(device)
X_pos  = batch.get("img_pos", None)
if X_pos is not None: 
    X_pos  = X_pos.to(device) # positive part for triplet loss
y      = batch["score"].to(device)
s_type = batch["score_type"]             # list[str]
s_type_np = np.array(s_type)
instance_label = np.array(batch["patient_scoretype_key"])


years_np = datestr_to_years_since_2000(batch["date_str"])
days_np = (years_np*365.2425).astype(int)
days_t = torch.from_numpy(days_np).to(device)
years_t = torch.from_numpy(years_np).to(device)


# B = X.size(0)

# s_type

In [ ]:
img_sample = X[0, 0].numpy(force=True)
plt.imshow(img_sample, cmap="gray")